In [1]:
import numpy as np
import znnl as nl

import pandas as pd

from flax import linen as nn
import optax

import matplotlib.pyplot as plt
from neural_tangents import stax

import h5py as hf
from scipy.stats import pearsonr

from rich.progress import track

import seaborn as sns

2023-08-10 14:22:28.725077: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Using backend: gpu

Available hardware:

gpu:0

## Download the data

In [2]:
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight',
                'Acceleration', 'Model Year', 'Origin']

raw_dataset = pd.read_csv(url, names=column_names,
                          na_values='?', comment='\t',
                          sep=' ', skipinitialspace=True)

dataset = raw_dataset.copy()
dataset = dataset.dropna()
dataset['Origin'] = dataset['Origin'].map({1: 'USA', 2: 'Europe', 3: 'Japan'})
dataset = pd.get_dummies(dataset, columns=['Origin'], prefix='', prefix_sep='')


dataset = (dataset-dataset.mean())/dataset.std()

class MPGDataGenerator(nl.data.DataGenerator):
    """
    Data generator for the MPG dataset.
    """
    def __init__(self, dataset: pd.DataFrame):
        """
        Constructor for the data generator.
        
        Parameters
        ----------
        dataset
        """        
        train_ds = dataset.sample(frac=0.8, random_state=0)
        train_labels = train_ds.pop("MPG")
        test_ds = dataset.drop(train_ds.index)
        test_labels = test_ds.pop("MPG")
        
        self.train_ds = {"inputs": train_ds.to_numpy(), "targets": train_labels.to_numpy().reshape(-1, 1)}
        self.test_ds = {"inputs": test_ds.to_numpy(), "targets": test_labels.to_numpy().reshape(-1, 1)}
        
        self.data_pool = self.train_ds["inputs"]

In [ ]:
ensembles = 20
optimizers = ["sgd", "traceopt", "adam"]

for i in range(ensembles):
    generator = MPGDataGenerator(dataset)
    seed = np.random.randint(low=0, high=564864168)

    for opt in optimizers:
        network = stax.serial(
            stax.Dense(128),
            stax.Relu(),
            stax.Dense(128),
            stax.Relu(),
            stax.Dense(1),
        )
        
        if opt == "traceopt":
            optimizer = nl.optimizers.TraceOptimizer(
                scale_factor=1.0, subset=0.2, memory=30
            )
        elif opt == "adam":
            optimizer = optax.adam(1e-2)
        elif opt == "sgd":
            optimizer = optax.sgd(1.0)
        else:
            raise NotImplementedError(
                "Optimizer doesn't exist..."
            )

        model = nl.models.NTModel(
                nt_module=network,
                optimizer=optimizer,
                seed=seed,
                input_shape=(1, 9),
            )

        train_recorder = nl.training_recording.JaxRecorder(
            name=f"{opt}_train_recorder_{i}",
            loss=True,
            update_rate=1
        )
        test_recorder = nl.training_recording.JaxRecorder(
            name=f"{opt}_test_recorder_{i}",
            loss=True,
            update_rate=1
        )

        train_recorder.instantiate_recorder(
            data_set=generator.train_ds
        )
        test_recorder.instantiate_recorder(
            data_set=generator.test_ds
        )

        training_strategy = nl.training_strategies.SimpleTraining(
            model=model, 
            loss_fn=nl.loss_functions.MeanPowerLoss(order=2),
            recorders=[train_recorder, test_recorder],
        )
        _ = training_strategy.train_model(
                train_ds=generator.train_ds,
                test_ds=generator.test_ds, 
                epochs=100, 
                batch_size=12,
            )

        train_recorder.dump_records()
        test_recorder.dump_records()

Epoch: 14:  14%|████▏                         | 14/100 [00:07<00:28,  3.06batch/s, test_loss=0.0866]

## Analysis

In [ ]:
def load_data(file):
    with hf.File(file, "r") as db:
        data = db["loss"][:]
        
    return data

In [ ]:
adam_data = []
adam_data_tr = []

for i in range(20):
    adam_data.append(
        load_data(f"adam_test_recorder_{i}.h5")
    )
    adam_data_tr.append(
        load_data(f"adam_train_recorder_{i}.h5")
    )

In [ ]:
traceopt_data = []
traceopt_data_tr = []

for i in range(20):
    traceopt_data.append(
        load_data(f"traceopt_test_recorder_{i}.h5")
    )
    traceopt_data_tr.append(
        load_data(f"traceopt_train_recorder_{i}.h5")
    )

In [ ]:
sgd_data = []
sgd_data_tr = []

for i in range(20):
    sgd_data.append(
        load_data(f"sgd_test_recorder_{i}.h5")
    )
    sgd_data_tr.append(
        load_data(f"sgd_train_recorder_{i}.h5")
    )

In [ ]:
x = np.linspace(1, 101, 100)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].errorbar(
    x,
    np.mean(traceopt_data, axis=0), 
    yerr=np.std(traceopt_data, axis=0),
    marker='x',
    c = "#140D4F",
    mfc="none",
    linestyle="none",
    label="TraceOpt"
)
ax[0].errorbar(
    x,
    np.mean(adam_data, axis=0), 
    yerr=np.std(adam_data, axis=0),
    marker='o',
    c="#4EA699",
    mfc="none",
    linestyle="none",
    label="ADAM"
)
ax[0].errorbar(
    x,
    np.mean(sgd_data, axis=0), 
    yerr=np.std(sgd_data, axis=0),
    marker='*',
    c="blue",
    mfc="none",
    linestyle="none",
    label="SGD"
)

ax[1].errorbar(
    x,
    np.mean(traceopt_data_tr, axis=0), 
    yerr=np.std(traceopt_data_tr, axis=0),
    marker='x',
    c = "#140D4F",
    mfc="none",
    linestyle="none",
    label="TraceOpt"
)
ax[1].errorbar(
    x,
    np.mean(adam_data_tr, axis=0), 
    yerr=np.std(adam_data_tr, axis=0),
    marker='o',
    c="#4EA699",
    mfc="none",
    linestyle="none",
    label="ADAM"
)
ax[1].errorbar(
    x,
    np.mean(sgd_data_tr, axis=0), 
    yerr=np.std(sgd_data_tr, axis=0),
    marker='*',
    c="blue",
    mfc="none",
    linestyle="none",
    label="SGD"
)

ax[0].set_xlabel("Epoch")
ax[0].set_ylabel(r"$\mathcal{L}_{test}$")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel(r"$\mathcal{L}_{train}$")

ax[0].set_yscale("log")
ax[0].set_xscale("log")
ax[1].set_yscale("log")
ax[1].set_xscale("log")

plt.legend()
plt.savefig("mpg-vs-adam.pdf")
plt.show()